# Citadel T1 — Calculator canary on Colab TPU
Self-contained T1 launcher. T0 is already certified and stored in the repository. Do not rerun T0 here. Flow: fresh Citadel + pinned Cymek runtime → TPU setup → T1 preflight → data receipt → preregistered train/eval → reload verification → export receipt.

In [ ]:
# 0. Fresh Citadel checkout + pinned read-only Cymek runtime
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('PJRT_DEVICE=' + str(os.environ.get('PJRT_DEVICE')))

In [ ]:
# A. T1 preflight. If this fails, STOP and send the full output.
import subprocess
p1 = subprocess.run(['python','-m','citadel_tpu.calculator_preflight'])
assert p1.returncode == 0, 'T1 PREFLIGHT failed — READY_FOR_T1=NO. STOP.'
print('READY_FOR_T1=YES')

In [ ]:
# B. Deterministic data receipt preview; no training yet.
from citadel_tpu import calculator_eval as cev
drc = cev.build_data_receipt()
print('counts:', drc['counts'])
print('overlap:', drc['overlap'])
print('scored slices:', drc['scored_slice_counts'])

In [ ]:
# C. Execute the frozen T1 protocol: baseline → dev-gated [5,20,100,200] ladder → final TEST → reload gate.
from citadel_tpu import calculator_train
r1 = calculator_train.train(out='docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print('T1 status:', r1['status'])
print('gate rules:', r1['gate_rules'])

In [ ]:
# D. Read the frozen result receipt; do not recompute or edit thresholds.
import json
receipt_path = 'docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json'
rc = json.load(open(receipt_path))
print('status:', rc['status'])
print('untrained:', rc['eval']['untrained_test'])
print('trained:', rc['eval']['trained_test'])
print('strongest null:', rc['strongest_heuristic_null'])
print('endpoint updates:', rc['training']['endpoint_updates'])

In [ ]:
# E. Reload identity check.
print('pre :', rc['pre_reload_prediction_sha256'])
print('post:', rc['post_reload_prediction_sha256'])
print('reload_identical:', rc['reload_identical'])

In [ ]:
# F. Export the exact T1 receipt.
from google.colab import files
files.download('docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print('Downloaded TPU_CALCULATOR_CHECKPOINT.json — upload this exact file back to ChatGPT.')